# Fase 3 · M04b: Encoding EDA (Mixto)

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 3 — Feature Engineering |
| **Módulo** | M04b — Encoding EDA |

---

## 🎯 Qué hace

Aplica encoding mixto (numérico + strings legibles) para EDA. Mantiene las categorías como texto para que los gráficos del EDA sean interpretables.

## 📋 Requisitos

- `data/03_features/df_expediente_features.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `data/03_features/df_exp_target_eda.parquet` | Dataset mixto para EDA |
| `docs/html/fase3/m04b_eda_target.html` | Informe HTML |

## 🔄 Flujo

```
df_expediente_features.parquet
    ↓ Encoding mixto (num + strings)
    ↓ Verificación de legibilidad
    → data/03_features/df_exp_target_eda.parquet + HTML
```

## ➡️ Siguiente

`f3_m04_index.ipynb` — verificación de ambos datasets de encoding


In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================

import sys
import warnings
from pathlib import Path
from datetime import datetime

warnings.filterwarnings('ignore')

# Detectar entorno
ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np

from src.config import RUTA_FEATURES, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# Rutas
RUTA_FASE3_HTML = RUTA_HTML / 'fase3'
crear_directorios([RUTA_FEATURES, RUTA_FASE3_HTML])

fmt = formato_numero_es
info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('\n' + '='*70)
print('F3-M04: ENCODING EDA (MIXTO - Numérico + Strings)')
print('='*70)

df = pd.read_parquet(RUTA_FEATURES / 'df_expediente_features.parquet')
n_registros_entrada = len(df)
n_cols_entrada = len(df.columns)

print(f'\n✓ Cargado: df_expediente_features.parquet')
print(f'  • Registros: {fmt(n_registros_entrada)}')
print(f'  • Columnas entrada: {n_cols_entrada}')


F3-M04: ENCODING EDA (MIXTO - Numérico + Strings)

✓ Cargado: df_expediente_features.parquet
  • Registros: 33.621
  • Columnas entrada: 51


In [3]:
# ============================================================================
# CELDA 3: MANTENER STRINGS - SIN CODIFICAR
# ============================================================================

print('\n' + '-'*70)
print('MANTENIENDO STRINGS (CatBoost maneja nativamente)')
print('-'*70)

strings_kept = []
for col in df.columns:
    if df[col].dtype == 'object':
        strings_kept.append(col)

if strings_kept:
    print(f'\n✓ Strings a mantener:')
    for col in sorted(strings_kept):
        n_unique = df[col].nunique()
        print(f'  • {col} ({n_unique} valores únicos)')
else:
    print('\n⚠️ No hay strings para mantener')

print(f'\n✓ Total strings: {len(strings_kept)}')


----------------------------------------------------------------------
MANTENIENDO STRINGS (CatBoost maneja nativamente)
----------------------------------------------------------------------

✓ Strings a mantener:
  • cupo (10 valores únicos)
  • egresado (2 valores únicos)
  • pais_nombre (77 valores únicos)
  • poblacion (933 valores únicos)
  • provincia (50 valores únicos)
  • rama (5 valores únicos)
  • sexo (2 valores únicos)
  • situacion_laboral (11 valores únicos)
  • titulacion (40 valores únicos)
  • universidad_origen (5 valores únicos)
  • via_acceso (13 valores únicos)

✓ Total strings: 11


In [4]:
# ============================================================================
# CELDA 4: PREPARAR DATOS NUMÉRICOS
# ============================================================================

print('\n' + '-'*70)
print('PREPARANDO DATOS NUMÉRICOS')
print('-'*70)

# Booleans → 0/1
bool_cols = df.select_dtypes(include='bool').columns.tolist()
for col in bool_cols:
    df[col] = df[col].astype(int)
print(f'\n✓ Booleans → 0/1 ({len(bool_cols)} cols)')

# Redondear floats
float_cols = df.select_dtypes(include=['float64', 'float32']).columns.tolist()
for col in float_cols:
    df[col] = df[col].round(2)
print(f'✓ Floats redondeados ({len(float_cols)} cols)')


----------------------------------------------------------------------
PREPARANDO DATOS NUMÉRICOS
----------------------------------------------------------------------

✓ Booleans → 0/1 (4 cols)
✓ Floats redondeados (20 cols)


In [5]:
# ============================================================================
# CELDA 5: ELIMINAR COLUMNAS
# ============================================================================

print('\n' + '-'*70)
print('ELIMINANDO COLUMNAS')
print('-'*70)

# fecha_nacimiento: redundante con edad_entrada
# poblacion: altísima cardinalidad — provincia suficiente
# cupo: se mantiene como string legible para EDA y CatBoost
# titulacion: se mantiene como string — CatBoost la maneja nativamente
#             (Opción G). M05 la usará para target encoding en AutoML.
cols_eliminar = ['fecha_nacimiento', 'poblacion']
cols_a_eliminar = [col for col in cols_eliminar if col in df.columns]

for col in cols_a_eliminar:
    df = df.drop(col, axis=1)
    print(f'✓ {col}')

print(f'\nTotal: {len(cols_a_eliminar)} eliminadas')


----------------------------------------------------------------------
ELIMINANDO COLUMNAS
----------------------------------------------------------------------
✓ fecha_nacimiento
✓ poblacion

Total: 2 eliminadas


In [6]:
# ============================================================================
# CELDA 6: VERIFICACIÓN Y ESTADÍSTICAS
# ============================================================================

print('\n' + '-'*70)
print('VERIFICACIÓN FINAL')
print('-'*70)

n_registros_salida = len(df)
n_cols_salida = len(df.columns)

strings = df.select_dtypes(include=['object']).columns.tolist()
numerics = df.select_dtypes(include=['number']).columns.tolist()

print(f'\n✓ Registros: {fmt(n_registros_salida)}')
print(f'✓ Columnas: {n_cols_salida}')
print(f'  • Strings: {len(strings)}')
print(f'  • Números: {len(numerics)}')

tipos = df.dtypes.value_counts()
print(f'\nTipos:')
for dtype, count in tipos.items():
    print(f'  • {dtype}: {count}')

print(f'\n✅ MIXTO - OK para CatBoost')


----------------------------------------------------------------------
VERIFICACIÓN FINAL
----------------------------------------------------------------------

✓ Registros: 33.621
✓ Columnas: 49
  • Strings: 10
  • Números: 39

Tipos:
  • float64: 20
  • int64: 19
  • object: 10

✅ MIXTO - OK para CatBoost


In [7]:
# ============================================================================
# CELDA 7: GUARDAR DATASET
# ============================================================================

print('\n' + '='*70)
print('GUARDANDO')
print('='*70)

ruta_salida = RUTA_FEATURES / 'df_exp_target_eda.parquet'
df.to_parquet(ruta_salida, index=False)
tamanio_mb = ruta_salida.stat().st_size / 1024 / 1024

print(f'\n✓ {ruta_salida.name}')
print(f'  • Registros: {fmt(n_registros_salida)}')
print(f'  • Columnas: {n_cols_salida}')
print(f'  • Tamaño: {tamanio_mb:.1f} MB')


GUARDANDO



✓ df_exp_target_eda.parquet
  • Registros: 33.621
  • Columnas: 49
  • Tamaño: 1.2 MB


In [8]:
# ============================================================================
# CELDA 8: GENERAR HTML
# ============================================================================

print('\n' + '='*70)
print('GENERANDO HTML')
print('='*70)

nav_fases_html, nav_modulos_html = generar_html_navegacion_completa(
    fase_activa='fase3',
    modulo_activo='m04'
)

# KPIs
KPIS = [
    {'valor': fmt(n_registros_salida), 'titulo': 'Expedientes'},
    {'valor': str(n_cols_salida), 'titulo': 'Columnas'},
    {'valor': f'{len(strings)}+{len(numerics)}', 'titulo': 'Mixto'},
    {'valor': '✓', 'titulo': 'CatBoost Listo'},
]
kpis_html = generar_kpis_html(KPIS)

# S1: Transformación
s1 = generar_seccion_html('Transformación', f'''
<div style="display:grid;grid-template-columns:1fr auto 1fr;gap:20px;align-items:center;text-align:center;">
    <div style="background:#ebf8ff;padding:20px;border-radius:10px;">
        <div style="font-size:28px;font-weight:bold;color:#3182ce;">{n_cols_entrada}</div>
        <div style="color:#2c5282;">columnas entrada</div>
    </div>
    <div style="font-size:48px;color:#a0aec0;">→</div>
    <div style="background:#f0fff4;padding:20px;border-radius:10px;">
        <div style="font-size:28px;font-weight:bold;color:#38a169;">{n_cols_salida}</div>
        <div style="color:#276749;">mixto (mantiene strings)</div>
    </div>
</div>
<p style="text-align:center;margin-top:15px;">Diferencia: Mantiene titulacion (STRING) vs M04a que la elimina</p>
''', '🔀')

# S2: Strings mantenidos
strings_html = ''
for col in sorted(strings):
    n_unique = df[col].nunique()
    strings_html += f'<div style="margin-bottom:8px;"><strong>{col}:</strong> {n_unique} valores</div>'

s2 = generar_seccion_html('Strings Mantenidos (CatBoost Nativo)', f'''
{strings_html}
<p style="margin-top:15px;padding:10px;background:#f7fafc;border-radius:5px;color:#4a5568;">
    CatBoost maneja categorías directamente - no requieren one-hot encoding
</p>
''', '🔤')

# S3: Estructura final
s3 = generar_seccion_html('Estructura Final', f'''
<div style="display:grid;grid-template-columns:repeat(3,1fr);gap:15px;">
    <div style="background:#fff5f5;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#e53e3e;">{len(strings)}</div>
        <div style="font-size:12px;color:#c53030;">object (strings)</div>
    </div>
    <div style="background:#ebf8ff;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#3182ce;">{sum(1 for t in df.dtypes if 'int' in str(t))}</div>
        <div style="font-size:12px;color:#2c5282;">int64</div>
    </div>
    <div style="background:#f0fff4;padding:15px;border-radius:8px;text-align:center;">
        <div style="font-size:24px;font-weight:bold;color:#38a169;">{sum(1 for t in df.dtypes if 'float' in str(t))}</div>
        <div style="font-size:12px;color:#276749;">float64</div>
    </div>
</div>
''', '📊')

contenido_html = kpis_html + s1 + s2 + s3

html_completo = render_pagina_desde_fichero(
    'f3_m04_eda_target.ipynb',
    contenido_html,
    carpeta_notebook='fase3_features'
)

ruta_html = RUTA_FASE3_HTML / 'm04_eda.html'
guardar_html(html_completo, ruta_html)
print(f'🌐 HTML: {ruta_html}')


GENERANDO HTML
✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m04_eda.html
🌐 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m04_eda.html


In [9]:
# ============================================================================
# CELDA 9: RESUMEN FINAL
# ============================================================================

print('\n' + '='*70)
print('✅ F3-M04 COMPLETADO')
print('='*70)
print(f'📥 Entrada: {n_cols_entrada} columnas')
print(f'📤 Salida: {n_cols_salida} columnas (MIXTO: {len(strings)} strings + {len(numerics)} números)')
print(f'💾 {ruta_salida}')
print(f'🌐 {ruta_html}')
print(f'\n📌 Siguiente: f3_m05_target_export.ipynb')


✅ F3-M04 COMPLETADO
📥 Entrada: 51 columnas
📤 Salida: 49 columnas (MIXTO: 10 strings + 39 números)
💾 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features\df_exp_target_eda.parquet
🌐 C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase3\m04_eda.html

📌 Siguiente: f3_m05_target_export.ipynb
